#### config

In [50]:
!pip install --upgrade pip
!pip install sentence-transformers faiss-cpu pandas numpy tqdm python-dotenv >> /dev/null

In [51]:
from __future__ import annotations
from pathlib import Path
import json, os, pickle, warnings, rich
from dataclasses import dataclass
from typing import Any, Literal, Optional
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import numpy.typing as npt
from sentence_transformers import SentenceTransformer
import faiss

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore',message='BertModel LOAD REPORT*')

#  position_ids.UNEXPECTED displays on loading the faiss index
#  can be ignored, since the model used is based on bert which requires this parameter;
#    but its configured since its a pretrained model

In [57]:
@dataclass()
class Config:
    data_dir = '/home/ronji/Dev/CarP/esco_dataset/formatted/'
    db_dir = 'db/'
    model_hf = 'simcse-dist-mpnet-paracrawl-cs-en'
    HF_TOKEN = os.getenv('HF_TOKEN')

    # embedding configuration
    batch_size = 256  # size of an encoding batch
    normalize_embeddings = True  # enable cosine similarity (inner product)
    index_type = 'flat'  # store and compare all vectors directly, used instead of IFV (index by similarity, for large datasets)


config = Config()

### Record class
record == ESCO entity

In [12]:
EntityType = Literal[
    'occupation',
    'skill',
    'isco_group',  # categorisation
    'skill_group'
]


@dataclass
class Label:
    entity_type: EntityType
    esco_uri: str
    id: str
    preferred_label: str
    alt_label: str
    description: str
    code: str
    isco_code: str

    @property
    def embedding_text(self) -> str:
        text = self.preferred_label
        if self.alt_label:
            text = '{} | {}'.format(text, self.alt_label)
        return text

    def to_json(self) -> dict[str, Any]:
        return {
            'entity_type': self.entity_type,
            'esco_uri': self.esco_uri,
            'id': self.id,
            'preferred_label': self.preferred_label,
            'alt_label': self.alt_label,
            'description': self.description,
            'code': self.code,
            'isco_code': self.isco_code
        }


### Data parsing

In [ ]:
def clean_text(value):
    text = str(value).replace('\xa0', ' ').strip() # most ESCO records contain both NBSP characters and newline
    return '' if pd.isna(value) else ' '.join(text.split())

def get_alt_labels(col_data):
    cleaned = clean_text(col_data)
    return [] if not cleaned \
        else [item for item in
              (label.strip() for label in cleaned.split('\n'))
              if item]

In [6]:
def load_occupation_data(occ_path) -> list[Label]:
    df = pd.read_csv(occ_path).fillna('')
    records = []
    for _,row in df.iterrows():
        alts = get_alt_labels(row.get('ALTLABELS'))
        if alts == []: alts.append('') # so that at least one label is added for each row
        for a in alts:
            records.append(Label(
                entity_type='occupation',
                esco_uri=row.get('ESCOURI'),
                id=row.get('ID'),
                preferred_label=row.get('PREFERREDLABEL'),
                alt_label=a,
                description=clean_text(row.get('DESCRIPTION')),
                code=row.get('CODE'),
                isco_code=row.get('ISCOGROUPCODE')
            ))
    return records


def load_skill_data(skill_path) -> list[Label]:
    df = pd.read_csv(skill_path).fillna('')
    records = []
    for _,row in df.iterrows():
        alts = get_alt_labels(row.get('ALTLABELS'))
        if alts == []: alts.append('')

        for a in alts:
            records.append(Label(
                entity_type='skill',
                esco_uri=row.get('ESCOURI'),
                id=row.get('ID'),
                preferred_label=row.get('PREFERREDLABEL'),
                alt_label=a,
                description=clean_text(row.get('DESCRIPTION')),
                code='',
                isco_code=''
            ))
    return records


def load_isco(isco_path) -> list[Label]:
    df: pd.DataFrame = pd.read_csv(isco_path).fillna('')
    records = []
    for _, row in df.iterrows():
        records.append(
            Label(
                entity_type='isco_group',
                esco_uri=row.get('ESCOURI'),
                id=row.get('ID'),
                preferred_label=row.get('PREFERREDLABEL'),
                alt_label='',
                description='',
                code=row.get('CODE'),
                isco_code=''
            )
        )
    return records


def load_skill_groups(sg_path) -> list[Label]:
    df: pd.DataFrame = pd.read_csv(sg_path, dtype=str, keep_default_na=False).fillna('')
    records = []
    for _, row in df.iterrows():
        records.append(
            Label(
                entity_type='skill_group',
                esco_uri=row.get('ESCOURI'),
                id=row.get('ID'),
                preferred_label=row.get('PREFERREDLABEL'),
                code=row.get('CODE'),
                alt_label='',
                description=clean_text(row.get('DESCRIPTION')),
                isco_code=''
            )
        )
    return records

In [33]:
occupations_path = config.data_dir + 'combinedOccupations_cs.csv'
skills_path = config.data_dir + 'combinedSkills_cs.csv'
iscogroups_path = config.data_dir + 'ISCOGroups_cs.csv'
skillgroups_path = config.data_dir + 'skillGroups_cs.csv'
labels: list[Label] = []

occ_labels = load_occupation_data(occupations_path)
labels.extend(occ_labels)

skill_labels = load_skill_data(skills_path)
labels.extend(skill_labels)

isco_labels = load_isco(iscogroups_path)
labels.extend(isco_labels)

skillgroup_labels = load_skill_groups(skillgroups_path)
labels.extend(skillgroup_labels)

print(f'labels:         {len(labels)}\n'
      f'occupations:    {len(occ_labels)}\n'
      f'skills:         {len(skill_labels)}\n'
      f'skill categories:       {len(skillgroup_labels)}\n'
      f'occupation categories:  {len(isco_labels)}')

labels:         20441
occupations:    3007
skills:         16175
skill categories:       640
occupation categories:  619


In [34]:
load_dotenv()

model = SentenceTransformer(model_name_or_path=config.model_hf,token=os.getenv('HF_TOKEN'))
embedding_dimension = model.get_sentence_embedding_dimension()

No sentence-transformers model found with name Seznam/retromae-small-cs. Creating a new one with mean pooling.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Create vector db

In [46]:
def create_index(label_source, db_name):
    # encode text segments into embeddings
    texts = [l.embedding_text for l in label_source]
    model = SentenceTransformer(model_name_or_path=config.model_hf, token=os.getenv('HF_TOKEN'))
    embeddings = model.encode(
        texts,
        batch_size=config.batch_size,
        show_progress_bar=True,
        normalize_embeddings=config.normalize_embeddings,  # enables cosine similarity
        convert_to_numpy=True
    )
    embeddings = embeddings.astype(np.float32) # faiss uses numpy types

    # build a database index file
    d = embeddings.shape[1] # TODO: notes
    index = faiss.IndexFlatIP(d)
    index.add(embeddings)
    faiss.write_index(index,f'{str(Path(config.db_dir))}/{db_name}.index')

    # map metadata to vectors
    metadata = [l.to_json() for l in label_source]
    with open(f'{config.db_dir}/{db_name}_metadata.json','w',encoding='utf-8') as w:
        json.dump(metadata,w,ensure_ascii=False,indent=4)

    # write piclkle files for id lookup option
    id_idx = {label.id: idx for idx, label in enumerate(label_source)}
    with open(f'{config.db_dir}/pickle/{db_name}_id-idx.pkl', 'wb') as f:
        pickle.dump(id_idx, f)

In [36]:
create_index(skill_labels,'skill')
create_index(occ_labels,'occupation')
create_index(isco_labels,'isco_group')
create_index(skillgroup_labels,'skill_group')

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

### Interface

In [55]:
def load_db(entity_type: EntityType = 'all'): # entry function to use in calls
    model = SentenceTransformer(config.model_hf, token=config.HF_TOKEN)
    index = faiss.read_index(f'{config.db_dir}{entity_type}.index')
    with open(f'{config.db_dir}{entity_type}_metadata.json', "r", encoding="utf-8") as f:
        metadata = json.load(f)

    return model, index, metadata

In [58]:
model,index,metadata = load_db('skill')

No sentence-transformers model found with name sentence-transformers/simcse-dist-mpnet-paracrawl-cs-en. Creating a new one with mean pooling.


OSError: sentence-transformers/simcse-dist-mpnet-paracrawl-cs-en is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [53]:
@dataclass
class Result:
    rank: int
    cosine_score: float # higher = more similar
    entity_type: str
    esco_uri: str
    id: str
    label: str
    code: str
    isco_code: str
    description: str


def search(
    query,
    model,
    index,
    metadata,
    top_n = 10,
):
    query_vector = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)

    scores: npt.NDArray[np.float32]
    indices: npt.NDArray[np.int64]
    results = []

    scores, indices = index.search(query_vector, top_n)
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1: continue # faiss for empty
        data = metadata[int(idx)]

        results.append(
            Result(
            rank=len(results) + 1,
            cosine_score=float(score),
            entity_type=data['entity_type'],
            label=data['preferred_label'],
            id=data['id'],
            code=data.get('code', ''),
            isco_code=data.get('isco_code',''),
            description=data.get('description','')
            )
        )
    return results


def print_results(results):
    for r in results:
        rich.print(r)
        print()

### testing examples

##### all

In [54]:
q = input('query all types: ')
results = search(query=q, top_n=5)
print_results(results)

No sentence-transformers model found with name Seznam/retromae-smll-cs. Creating a new one with mean pooling.


OSError: Seznam/retromae-smll-cs is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

##### occupations

In [ ]:
q_occ = input('query occupations: ')
results = search(q_occ, top_n=3, entity_type='occupation')
results_isco = search(q_occ,top_n=3, entity_type='isco_group')

print_results(results)
print_results(results_isco)

##### skills

In [ ]:
q_s = input('query skills: ')
results = search(q_s, top_n=3, entity_type='skill')
results_groups = search(q_s,top_n=3, entity_type='skill_group')

print_results(results)
print_results(results_groups)